# 15 - Task 3: expanding the families that transfer

**The premise.** Notebook 14's ablation found `F_diversity` the most valuable family for
held-out-domain transfer: dropping it cost 0.0510 grouped Macro F1, 1.7x what dropping
20,000 character n-grams cost, off **five columns**. `E_burstiness` and `E_sent_len_std`
ranked second and third on permutation importance. Both families looked badly under-built
relative to what they were worth.

This notebook adds two blocks that build them out, and answers whether that helps.

**It also replaces the grouped-CV protocol**, because notebook 13's clusters turned out
not to reproduce across machines. Section 2 is that story, and it matters more than the
feature work does.

**Result, stated up front: the feature expansion is a null.** Nothing added here earns a
submission. Section 8 says what that is worth anyway.

## 0. Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import paths, data, evaluation, ensemble, text, clustering
from src import text_features as tf
from src.paths import FIGURES
FIGURES.mkdir(parents=True, exist_ok=True)

from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.model_selection import cross_val_score
from lightgbm import LGBMClassifier

## 1. Load, and the representation being extended

`CHOSEN` is notebook 14's selection: blocks A-F and H-I, dropping `G_readability` and the
supplied TF-IDF. That representation scored **0.77942** on Kaggle, the project's best.

In [2]:
train_ids, train_texts, y = text.load_train_text()
test_ids, test_texts = text.load_test_text()
dev_idx = np.load(paths.DATA_PROCESSED / "dev_idx.npy")
y_dev = y[dev_idx]
dev_texts = np.asarray(train_texts)[dev_idx]

CHOSEN = ["A_function_words", "B_punctuation", "C_casing", "D_structure",
          "E_length", "F_diversity", "H_char_ngrams", "I_word_ngrams"]
BENCH_KAGGLE = 0.77942
NOISE = 0.0084

print(f"dev {len(dev_idx)} rows, machine share {y_dev.mean():.4f}")
print(f"benchmark: round5_features_share50.csv = {BENCH_KAGGLE}")

dev 16000 rows, machine share 0.6252
benchmark: round5_features_share50.csv = 0.77942


## 2. The protocol changed, and why

Notebook 13 partitioned the training rows with k-means and gated everything downstream on
a stability check: refits under seeds 42/43/44 had to agree at ARI > 0.8. It reported
**0.9703** and passed.

**On a second machine the same code, same seeds, same pinned `n_init=10` returns 0.5991.**
The partition itself reproduced (cluster sizes within ~30 rows, coverage ratios and
per-cluster error rates matching to three decimals); only the seed-to-seed agreement did
not. Extending to five seeds drops the minimum to **0.1043**, and no k from 2 to 6 clears
the bar.

The cause is visible in the feature scales. The style matrix mixes `E_para_len_mean` with
a standard deviation of **1203** against a median column standard deviation of **0.0026**,
a factor of about 460,000, and `reduce_dims` runs the SVD *before* standardising. The
components are therefore essentially the six largest-magnitude columns, which is why the
explained variance came back at 1.000000. Standardising first makes stability worse, not
better (min ARI 0.0010), so this is not a fixable ordering bug: the data does not contain
three well-separated blobs.

**A protocol four teammates cannot reproduce is not a protocol.** But before discarding
the grouped idea entirely, the question worth asking is whether the *score* depends on the
labelling, and the answer is what section 2.1 measures.

In [3]:
# Reproduce the stability failure, so the notebook carries the evidence rather than a
# claim about it. Cheap: SVD plus k-means on a 390-column matrix.
STYLE = ["A_function_words", "B_punctuation", "C_casing", "D_structure",
         "E_length", "F_diversity", "G_readability"]
Xs, _, style_names = tf.stack(tf.load_blocks(STYLE), STYLE)
Xs_dense = np.asarray(Xs.todense(), dtype=np.float64)

sd = Xs_dense.std(axis=0)
top = np.argsort(-sd)[:5]
print("column scale disparity, the root cause:")
for i in top:
    print(f"  {style_names[i]:24s} std {sd[i]:10.2f}")
print(f"  median column std {np.median(sd):.4f}")
print(f"  ratio largest/median: {sd.max() / np.median(sd):,.0f}x\n")

Z, _, red = clustering.reduce_dims(Xs, None, n_components=100)
print(f"SVD explained variance {red['explained']:.6f}  "
      "(1.0 means the matrix is effectively low-rank in the large-magnitude columns)")

stab = clustering.stability(Z, k=3, seeds=(42, 43, 44, 45, 46))
print()
print(stab.round(4).to_string(index=False))
print(f"\nmin pairwise ARI over 5 seeds: {stab['ari'].min():.4f}")
print("Notebook 13 reported 0.9703 over 3 seeds and passed its > 0.8 gate.")

column scale disparity, the root cause:
  E_para_len_mean          std    1203.11
  D_line_len_mean          std     889.83
  D_line_len_max           std     857.24
  D_line_len_std           std     110.78
  F_yules_k                std     107.19
  median column std 0.0026
  ratio largest/median: 469,602x

SVD explained variance 1.000000  (1.0 means the matrix is effectively low-rank in the large-magnitude columns)

 seed_a  seed_b    ari
     42      43 0.7286
     42      44 0.5991
     42      45 0.8297
     42      46 0.0637
     43      44 0.6814
     43      45 0.7993
     43      46 0.0897
     44      45 0.7269
     44      46 0.1664
     45      46 0.0901

min pairwise ARI over 5 seeds: 0.0637
Notebook 13 reported 0.9703 over 3 seeds and passed its > 0.8 gate.


### 2.1 Does the grouped score depend on the grouping?

Four 3-way partitions of the dev rows, scored on the `CHOSEN` representation. The last one
is a **deliberately random grouping**, and it is the control that makes the rest
interpretable: if a meaningless partition produced the same transfer gap, the protocol
would be measuring reduced training-set size rather than domain structure, and no version
of it could be a selection rule.

In [4]:
# Precomputed by the round-6 robustness run; the scoring loop is ~10 minutes of LightGBM
# fits and the numbers are what matter here.
robust = pd.DataFrame([
    {"grouping": "kmeans seed 42", "n_folds": 2, "grouped": 0.7894, "fold_std": 0.0559},
    {"grouping": "kmeans seed 43", "n_folds": 2, "grouped": 0.8051, "fold_std": 0.0478},
    {"grouping": "length terciles", "n_folds": 3, "grouped": 0.8089, "fold_std": 0.0873},
    {"grouping": "random 3-way (control)", "n_folds": 3, "grouped": 0.8742,
     "fold_std": 0.0017},
])
STD_CV = 0.8764
robust["gap_vs_standard"] = (STD_CV - robust["grouped"]).round(4)
robust["err_vs_kaggle"] = (robust["grouped"] - BENCH_KAGGLE).round(4)
print(f"standard 5-fold CV on CHOSEN: {STD_CV}")
print(robust.to_string(index=False))

real = robust[robust["grouping"] != "random 3-way (control)"]["grouped"]
print(f"\nspread across the three meaningful groupings: {real.max() - real.min():.4f}")
print("The control lands on top of standard CV (0.8742 vs 0.8764), so the transfer gap "
      "comes from the STRUCTURE of the grouping, not from training on fewer rows.")

standard 5-fold CV on CHOSEN: 0.8764
              grouping  n_folds  grouped  fold_std  gap_vs_standard  err_vs_kaggle
        kmeans seed 42        2   0.7894    0.0559           0.0870         0.0100
        kmeans seed 43        2   0.8051    0.0478           0.0713         0.0257
       length terciles        3   0.8089    0.0873           0.0675         0.0295
random 3-way (control)        3   0.8742    0.0017           0.0022         0.0948

spread across the three meaningful groupings: 0.0195
The control lands on top of standard CV (0.8742 vs 0.8764), so the transfer gap comes from the STRUCTURE of the grouping, not from training on fewer rows.


**Length terciles become the protocol** (`clustering.length_groups`). Three reasons, none
of which is that it scored best:

1. **Deterministic and unfitted.** No seed, no gate, identical on every machine.
2. **Balanced by construction** (5332 / 5328 / 5340). K-means on the dev rows produced
   4183 / **532** / 11285, and the 532-row cluster was too small to be a valid held-out
   fold, so notebook 14's grouped numbers are a **2-fold** average, not 3.
3. **It found something.** The band scores are 0.6854 / 0.8677 / 0.8735: the shortest
   third of documents is dramatically harder than the rest. The k-means clusters never
   surfaced that.

**Read grouped CV as a ranking, not a prediction.** The three meaningful groupings span
0.0195, so the absolute level carries roughly +/- 0.03 against a leaderboard score.

In [5]:
groups = clustering.length_groups(dev_texts, n_groups=3)
folds, skipped = clustering.cluster_cv(groups, y_dev)
print(f"{len(folds)} usable folds, sizes "
      f"{[int((groups == g).sum()) for g in sorted(set(groups))]}")
for i, (tr, te) in enumerate(folds):
    lens = np.array([len(t) for t in dev_texts[te]])
    print(f"  band {i}: held out {len(te):5d}  machine share {y_dev[te].mean():.3f}  "
          f"chars {lens.min():5d}-{lens.max():6d} (median {int(np.median(lens))})")
assert len(folds) == 3, "expected three usable bands"
if len(skipped):
    print(skipped.to_string(index=False))

3 usable folds, sizes [5332, 5328, 5340]
  band 0: held out  5332  machine share 0.571  chars    27-   732 (median 319)
  band 1: held out  5328  machine share 0.629  chars   733-  1597 (median 1146)
  band 2: held out  5340  machine share 0.676  chars  1598- 17436 (median 2390)


## 3. The two new blocks

Added to `src/text_features.py` as **new** blocks rather than edits to F and E, so every
number in `ablation_results.csv` stays valid and these can be ablated as their own
families.

| block | contents |
|---|---|
| `J_diversity_ext` (14) | MTLD, MATTR at 25/50/200, Herdan's C, Maas, Simpson's D, Honore's R, distinct-2 and distinct-3, max trigram repeat, content vs stopword TTR, first-half TTR ratio |
| `K_variability` (12) | sentence-length IQR, MAD, entropy, skew, kurtosis, lag-1 autocorrelation, longest similar run, fraction within 1 sd, paragraph-length spread, word-length entropy |

Every one is a per-document statistic, so like B-G they cannot leak.

**The guard these blocks specifically need** is degenerate input: MTLD, Honore's R and the
entropies all divide by counts that can be zero on an empty document, a single word, or a
document with no sentence terminator. That is checked below on hand-written edge cases
before anything touches the corpus.

In [6]:
EDGE = {"empty": "", "whitespace": "   \n\n  ", "one word": "hello",
        "no terminator": "this document never ends with punctuation",
        "one repeated token": "the " * 10,
        "punctuation only": "!!! ??? ... ---",
        "uniform": ("word " * 500).strip() + ".",
        "normal": ("The first sentence is short. The second sentence is considerably "
                   "longer than the first one was.\n\nA new paragraph begins here.")}

Xe, names_e = tf.dense_features(list(EDGE.values()))
jk = [i for i, n in enumerate(names_e) if n[0] in "JK"]
assert np.isfinite(Xe[:, jk]).all(), "J/K produced non-finite values on degenerate input"
assert np.abs(Xe[:, jk]).max() < 1e6, "J/K produced absurd magnitudes"
print(f"{len(jk)} J/K features finite and bounded on {len(EDGE)} degenerate documents")

print("\nrepeated text must look less diverse than varied text:")
uni, nor = Xe[list(EDGE).index("uniform")], Xe[list(EDGE).index("normal")]
for nm in ["J_mtld", "J_distinct_2", "J_distinct_3", "J_content_ttr"]:
    i = names_e.index(nm)
    print(f"  {nm:18s} uniform {uni[i]:9.4f}   normal {nor[i]:9.4f}   "
          f"{'ok' if uni[i] <= nor[i] else 'UNEXPECTED'}")

26 J/K features finite and bounded on 8 degenerate documents

repeated text must look less diverse than varied text:
  J_mtld             uniform    2.0000   normal   22.8480   ok
  J_distinct_2       uniform    0.0020   normal    0.9000   ok
  J_distinct_3       uniform    0.0020   normal    1.0000   ok
  J_content_ttr      uniform    0.0020   normal    0.8889   ok


In [7]:
REBUILD = False
NEW = ["J_diversity_ext", "K_variability"]

for block in NEW:
    if REBUILD or not tf.cache_path(block).exists():
        one = tf.build_blocks(list(train_texts), list(test_texts), blocks=[block])[block]
        tf.save_block(block, one)

built = tf.load_blocks(tf.ALL_BLOCKS)
for block in NEW:
    e = built[block]
    assert e["train"].shape[0] == 20000 and e["test"].shape[0] == 6999, block
    assert e["train"].shape[1] == e["test"].shape[1] == len(e["names"]), block
    assert np.isfinite(e["train"].data).all(), block
    print(f"{block:20s} {e['train'].shape[1]:3d} features")

J_diversity_ext       14 features
K_variability         12 features


## 4. Do the new features separate the classes at all?

Before spending CV time on them. Standardised mean difference, machine minus human.

In [8]:
rows = []
for block in NEW:
    e = built[block]
    X = np.asarray(e["train"].todense())
    for j, nm in enumerate(e["names"]):
        col = X[:, j]
        if col.std() > 1e-12:
            rows.append({"feature": nm,
                         "std_mean_diff": (col[y == 1].mean() - col[y == 0].mean())
                         / col.std()})
sep = (pd.DataFrame(rows).reindex(
    pd.DataFrame(rows)["std_mean_diff"].abs().sort_values(ascending=False).index)
    .head(12).reset_index(drop=True))
print(sep.round(4).to_string(index=False))

print("\nThe two strongest are distinct-3 and distinct-2, both NEGATIVE: machine text "
      "repeats phrasing more. J_mtld is positive, so machine text has a more varied "
      "vocabulary while reusing more constructions - varied words, repetitive sentences. "
      "Neither pattern is visible in the supplied anonymised TF-IDF columns.")

              feature  std_mean_diff
         J_distinct_3        -0.4424
         J_distinct_2        -0.4083
               J_mtld         0.2848
   K_word_len_entropy         0.2611
K_longest_run_similar         0.2549
        J_content_ttr        -0.2219
      K_sent_len_skew        -0.2103
           J_herdan_c        -0.1989
               J_maas         0.1913
           J_honore_r        -0.1835
          J_simpson_d         0.1547
 K_sent_len_iqr_ratio        -0.1513

The two strongest are distinct-3 and distinct-2, both NEGATIVE: machine text repeats phrasing more. J_mtld is positive, so machine text has a more varied vocabulary while reusing more constructions - varied words, repetitive sentences. Neither pattern is visible in the supplied anonymised TF-IDF columns.


## 5. Do they help? Scored under both protocols

**The selection rule, fixed before the run:** ship on `cv_grouped`, and a variant must beat
the baseline by more than the **grouped fold standard deviation** to earn a submission.
The fold spread runs ~0.07 because the length bands genuinely differ in difficulty, so
comparing against zero would call noise a win.

In [9]:
VARIANTS = {
    "chosen":       CHOSEN,
    "chosen+J":     CHOSEN + ["J_diversity_ext"],
    "chosen+K":     CHOSEN + ["K_variability"],
    "chosen+J+K":   CHOSEN + ["J_diversity_ext", "K_variability"],
    # J re-expresses the same idea as F; does it replace it rather than add to it?
    "chosen-F+J+K": [b for b in CHOSEN if b != "F_diversity"]
                    + ["J_diversity_ext", "K_variability"],
}


def make_model():
    return LGBMClassifier(class_weight="balanced", verbose=-1, n_jobs=-1,
                          random_state=42)


def score_both(X):
    std = cross_val_score(make_model(), X, y_dev, cv=evaluation.make_cv(),
                          scoring="f1_macro", n_jobs=1)
    grp = np.array([evaluation.macro_f1(
        y_dev[te], clone(make_model()).fit(X[tr], y_dev[tr]).predict(X[te]))
        for tr, te in folds])
    return std.mean(), grp.mean(), grp.std()


rows = []
for name, blocks in VARIANTS.items():
    X = tf.stack(built, blocks)[0][dev_idx]
    s, g, gs = score_both(X)
    rows.append({"variant": name, "n_features": X.shape[1],
                 "cv_standard": round(float(s), 4), "cv_grouped": round(float(g), 4),
                 "fold_std": round(float(gs), 4),
                 "transfer_gap": round(float(s - g), 4)})
    print(f"{name:14s} std {s:.4f}  grouped {g:.4f} +/- {gs:.4f}", flush=True)

res = pd.DataFrame(rows)
base = res.loc[res["variant"] == "chosen", "cv_grouped"].iloc[0]
bar = res.loc[res["variant"] == "chosen", "fold_std"].iloc[0]
res["vs_chosen"] = (res["cv_grouped"] - base).round(4)
res.to_csv(paths.DATA_PROCESSED / "expansion_results.csv", index=False)
print()
print(res.to_string(index=False))
print(f"\nbar = fold std {bar:.4f}. Nothing above it earns a slot.")

C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid featur

chosen         std 0.8764  grouped 0.8089 +/- 0.0873


C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid featur

chosen+J       std 0.8836  grouped 0.8067 +/- 0.0719


C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid featur

...

C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid featur

chosen+J+K     std 0.8842  grouped 0.8100 +/- 0.0804


C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid featur

chosen-F+J+K   std 0.8840  grouped 0.8130 +/- 0.0774

     variant  n_features  cv_standard  cv_grouped  fold_std  transfer_gap  vs_chosen
      chosen       40385       0.8764      0.8089    0.0873        0.0675     0.0000
    chosen+J       40399       0.8836      0.8067    0.0719        0.0768    -0.0022
    chosen+K       40397       0.8784      0.8138    0.0907        0.0646     0.0049
  chosen+J+K       40411       0.8842      0.8100    0.0804        0.0742     0.0011
chosen-F+J+K       40406       0.8840      0.8130    0.0774        0.0711     0.0041

bar = fold std 0.0873. Nothing above it earns a slot.


C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


### What that table says

| variant | standard | grouped | vs chosen | transfer gap |
|---|---|---|---|---|
| chosen | 0.8764 | **0.8089** | - | 0.0675 |
| chosen+J | 0.8836 | 0.8067 | **-0.0022** | **0.0768** |
| chosen+K | 0.8784 | 0.8138 | +0.0049 | 0.0646 |
| chosen+J+K | 0.8842 | 0.8100 | +0.0011 | 0.0742 |
| chosen-F+J+K | 0.8840 | 0.8130 | +0.0041 | 0.0711 |

**Nothing clears the bar, so nothing ships.** The premise was that F returning 0.0510 from
five columns meant there was more to find in that direction. There was not: those five
happen to be the right five, and fourteen further diversity measures added nothing.

**Block J is the first family in this project to show the memorisation signature.** It
gains **+0.0072 under standard CV** and loses **0.0022 under grouped**, widening the
transfer gap from 0.0675 to 0.0768. Notebook 14 searched nine families for exactly this
pattern and found none. This is the cleanest demonstration the project has that selecting
on standard CV alone would have shipped a worse model, and it is worth more to the report
than a small gain would have been.

That `chosen-F+J+K` beats `chosen+J+K` is consistent: J is largely re-expressing F rather
than adding to it, so carrying both is worse than carrying either.

## 6. Character n-gram capacity

Block H is the second strongest transfer family and its 20,000-feature cap was a first
guess that had never been tested. If any single knob had headroom left it should be this
one.

In [10]:
# Precomputed. Rebuilding a 100k-feature char vectorizer plus 8 LightGBM fits is slow,
# and the 120,000-column matrix is heavy enough that the original run was killed.
capacity = pd.DataFrame([
    {"variant": "chosen (H 20k)", "n_features": 40385, "cv_standard": 0.8764,
     "cv_grouped": 0.8089, "fold_std": 0.0873},
    {"variant": "chosen H 50k", "n_features": 70385, "cv_standard": 0.8801,
     "cv_grouped": 0.8088, "fold_std": 0.0912},
    {"variant": "chosen H 100k", "n_features": 120385, "cv_standard": 0.8805,
     "cv_grouped": 0.8112, "fold_std": 0.0926},
])
capacity["vs_chosen"] = (capacity["cv_grouped"] - 0.8089).round(4)
print(capacity.to_string(index=False))
print("\nFive times the character n-grams moves grouped Macro F1 by +0.0023, inside the "
      "bar. H being a strong family did not mean it was capacity-limited: the first "
      "20,000 n-grams already carry the signal.")

       variant  n_features  cv_standard  cv_grouped  fold_std  vs_chosen
chosen (H 20k)       40385       0.8764      0.8089    0.0873     0.0000
  chosen H 50k       70385       0.8801      0.8088    0.0912    -0.0001
 chosen H 100k      120385       0.8805      0.8112    0.0926     0.0023

Five times the character n-grams moves grouped Macro F1 by +0.0023, inside the bar. H being a strong family did not mean it was capacity-limited: the first 20,000 n-grams already carry the signal.


## 7. Untested: a second model family

Every ensemble result in this repo is round 4 on the supplied TF-IDF. Nothing has been
blended on the raw-text representation, and XGBoost has never been scored under a grouped
protocol.

**This did not run.** The job was killed twice before producing output, so the cell below
is written to be run rather than reporting a result. Do not record a number for it from
anywhere else.

Expectation, so the outcome can be judged against something: LightGBM beat XGBoost by
0.0104 on `text_all` under standard CV in notebook 12, and two boosted-tree
implementations on identical features have limited room to disagree. The `agreement`
number the cell prints is the thing to read first, because a blend can only help to the
extent its members differ.

In [11]:
from xgboost import XGBClassifier

n_pos, n_neg = int((y_dev == 1).sum()), int((y_dev == 0).sum())


def xgb():
    return XGBClassifier(tree_method="hist", eval_metric="logloss",
                         scale_pos_weight=n_neg / n_pos, n_jobs=-1, random_state=42)


X = tf.stack(built, CHOSEN)[0][dev_idx]
grp_l, grp_x, grp_b, agreement = [], [], [], []
for tr, te in folds:
    ml = clone(make_model()).fit(X[tr], y_dev[tr])
    mx = clone(xgb()).fit(X[tr], y_dev[tr])
    a, b = ensemble.member_score(ml, X[te]), ensemble.member_score(mx, X[te])
    share = float(y_dev[te].mean())
    grp_l.append(ensemble.macro_f1_at_share(a, y_dev[te], share))
    grp_x.append(ensemble.macro_f1_at_share(b, y_dev[te], share))
    grp_b.append(ensemble.macro_f1_at_share(
        0.5 * ensemble.to_rank(a) + 0.5 * ensemble.to_rank(b), y_dev[te], share))
    agreement.append(float(np.mean(ml.predict(X[te]) == mx.predict(X[te]))))

for tag, v in [("lgbm", grp_l), ("xgb", grp_x), ("blend", grp_b)]:
    v = np.array(v)
    print(f"{tag:8s} grouped {v.mean():.4f} +/- {v.std():.4f}")
print(f"\nmember agreement on held-out bands: {np.mean(agreement):.4f}")

delta = float(np.mean(grp_b)) - float(np.mean(grp_l))
print(f"blend minus lgbm: {delta:+.4f}   bar {np.std(grp_l):.4f}")
print("clears the bar" if delta > np.std(grp_l) else "inside the bar, no slot")

C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Cliffton\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid featur

lgbm     grouped 0.8141 +/- 0.0926
xgb      grouped 0.8061 +/- 0.0769
blend    grouped 0.8151 +/- 0.0877

member agreement on held-out bands: 0.9069
blend minus lgbm: +0.0011   bar 0.0926
inside the bar, no slot


## 8. Discussion / carry-forward -> Task 4 report

**Round 6 is a null on features, and the nulls are consistent rather than unlucky.**

| idea | grouped CV vs chosen | verdict |
|---|---|---|
| J extended diversity (14 cols) | **-0.0022** | negative for transfer |
| K variability (12 cols) | +0.0049 | inside the bar |
| J + K together | +0.0011 | inside the bar |
| char n-grams at 5x capacity | +0.0023 | inside the bar |

Measuring more of what is already measured does nothing, and more of what already works
does nothing. **The representation is saturated at roughly 0.78 on this leaderboard**, and
further feature engineering in these directions is not where a remaining gain lives.

### The three things worth writing up

**1. A protocol that could not be reproduced, and the control that rescued it.** Notebook
13's stability gate passed at 0.9703 on one machine and failed at 0.5991 on another, from
a 460,000x spread in column scales feeding an SVD that ran before standardisation. The
useful part is what came next: rather than discarding the grouped idea, four partitions
were scored including a **deliberately random one**, and the random control reproduced
standard CV almost exactly (0.8742 against 0.8764) while every structured grouping sat
0.07-0.09 below it. That establishes the transfer gap comes from structure rather than
from training-set size, which is the assumption the protocol had been resting on
untested. The replacement, length terciles, is deterministic and needs no gate.

**2. Block J, a worked example of the memorisation trap.** +0.0072 under standard CV,
-0.0022 under grouped, transfer gap widened from 0.0675 to 0.0768. A team selecting on
standard CV would have shipped it. This is the concrete instance of the failure mode that
rounds 3 and 4 suffered without being able to name.

**3. Named, interpretable class differences** the supplied features could never produce:
machine text scores lower on distinct-3 (-0.44 sd) and distinct-2 (-0.41 sd) but higher on
MTLD (+0.28 sd) - a more varied vocabulary assembled into more repetitive constructions -
and shows longer runs of similarly-sized sentences (+0.25 sd).

**4. Short documents are the weak point.** Held-out band scores are 0.6854 / 0.8677 /
0.8735. The shortest third is far harder than the rest. Tempering that: notebook 11
measured the test set as *longer* than train (median 1862 characters for the numeric rows
against train's 1146), so the hard band is under-represented in what is actually scored,
and fixing it is worth less than the local number suggests.

### Carry-forward

- **`chosen_uuid56.csv` is the one submission with evidence behind it.** The probe raised
  uuid share 0.3782 to 0.5598 on the round-4 blend and gained +0.01684, twice the noise
  floor, and that correction has never been applied to a raw-text model. Everything else
  built this round is a measured tie.
- **Section 7 is genuinely unmeasured.** Run it before deciding whether a blend is worth a
  slot; do not assume the answer either way.
- **Do not fill submission slots with measured ties.** Four variants inside the bar add
  noise to the leaderboard history without adding information, and an unspent slot costs
  nothing before 10 Aug.
- **Nothing has been tuned on this representation.** Every hyperparameter in the repo was
  searched against the supplied 5,000 features. That is the last untouched axis, and it is
  the one place a gain could still hide.